In [2]:
import os
from dotenv import load_dotenv
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [3]:

from langchain_groq import ChatGroq

model=ChatGroq(model_name="llama-3.3-70b-versatile")
model



ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002157C280830>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002157C281550>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [9]:
from langchain.tools import tool

@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

@tool
def subtract_numbers(a: int, b: int) -> int:
    """Subtract one number from another."""
    return a - b


model_with_tools=model.bind_tools([add_numbers, subtract_numbers])

# now i will invoke the model with a question that requires the use of the tool
response=model_with_tools.invoke("What is the sum of 5 and 7 and subtract 3?")
print(response)

# ab is response ko print kiya to you can see ki iske andr na ek tool_calls krke hai jo store krta hai ki LLM n kon si tool call ki hai..and us tool call me kya argument pass ki hui hai
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")
   





content='' additional_kwargs={'tool_calls': [{'id': 'pny69fdgw', 'function': {'arguments': '{"a":5,"b":7}', 'name': 'add_numbers'}, 'type': 'function'}, {'id': 'g7mx8pvh4', 'function': {'arguments': '{"a":12,"b":3}', 'name': 'subtract_numbers'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 290, 'total_tokens': 328, 'completion_time': 0.100325942, 'completion_tokens_details': None, 'prompt_time': 0.149026511, 'prompt_tokens_details': None, 'queue_time': 1.30838539, 'total_time': 0.249352453}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e914b-067b-7451-98b4-023f962b93a7-0' tool_calls=[{'name': 'add_numbers', 'args': {'a': 5, 'b': 7}, 'id': 'pny69fdgw', 'type': 'tool_call'}, {'name': 'subtract_numbers', 'args': {'a': 12, 'b': 3}, 'id': 'g7mx8pvh4', 'type': 'tool_call'}] invalid_tool_

In [14]:
### above this is happening
### You ask: "What is the sum of 5 and 7?"
### The LLM responds with a tool_calls entry saying: "call add_numbers with a=5, b=7"
### Missing step: You never actually invoke the tool function.

In [10]:
# To get the actual result, you need to execute the tool yourself:

# niche bata rkha hu why..
tools_available={"add_numbers": add_numbers, "subtract_numbers": subtract_numbers}
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")
    
    # Actually execute the tool

    # par ye to hardcoded ho gya ki add_number hi bula rhe
    # result = add_numbers.invoke(tool_call['args'])

    # chuki llm has already decided ki add_numbers tool ko call karna hai, to hum directly usko invoke kar dete hai with the arguments provided by the LLM
    # and ye add_number to tool_call['name'] se match kar raha hai, to hum directly usko invoke kar dete hai with the arguments provided by the LLM
    # result=tool_call['name'].invoke(tool_call['args'])
    # but above line will give error kyuki tool_call['name'] is a string, not the actual function. So we need to map the string to the actual function.
    
    # isliye upar kahi pe hame add_numbers aur subtract_numbers ko define karna pada tha, taki hum unko map kar sakein with the string name of the tool.
    # like this: tools_available={"add_numbers": add_numbers, "subtract_numbers": subtract_numbers}

    # and phir jab hum let tool_call['Name'] me let "add_number" aya to phir ham tool_available["add_number"] se --add_number function invoke kar sakte hai with the arguments provided by the LLM.

    
    result = tools_available[tool_call['name']].invoke(tool_call)

    # so you might be  wondering ki ye invoke kaise karna hai, to you can see that tool_call me ek "args" field hota hai jisme arguments store hote hai jo LLM ne provide kiye hai. To hum usko directly pass kar dete hai invoke function me.
    # but above hmne to .invoke(tool_call) kr diya pr hame parameter pass krna tha to hame to .invoke(tool_call['args'] ) krna chaiye tha na...par dono sahi hai
    # invoke function internally deconstructs the arguments from the tool_call object, so you can pass the entire tool_call object or just the args, both will work. But it's more common to just pass the args directly.

    # so this below line of code is also correct
    # result = tools_available[tool_call['name']].invoke(tool_call['args'])

    

    # to you can see the output ki phela loop chala to add kiya and then substract kiya
    print(f"Result: {result}")

Tool called: add_numbers
Arguments: {'a': 5, 'b': 7}
Result: content='12' name='add_numbers' tool_call_id='pny69fdgw'
Tool called: subtract_numbers
Arguments: {'a': 12, 'b': 3}
Result: content='9' name='subtract_numbers' tool_call_id='g7mx8pvh4'


In [ ]:

# isme ham dikha rhe hai ki kaise ham user message se start krke, tool call ko execute krke, aur phir uska result leke wapas user ko de sakte hai.
# phir we can see that jab first question pucha to ...and response aya vo AI trf se aya..usme koi tool invoke nhi hua tha..bas decide kiya tha kon sa tool call krna hai

messages=[{"role": "user", "content": "What is the sum of 5 and 7 and subtract 3?"}]
response=model_with_tools.invoke(messages)
messages.append(response)

for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")
    
    # Actually execute the tool

    # par ye to hardcoded ho gya ki add_number hi bula rhe
    # result = add_numbers.invoke(tool_call['args'])

    # chuki llm has already decided ki add_numbers tool ko call karna hai, to hum directly usko invoke kar dete hai with the arguments provided by the LLM
    # and ye add_number to tool_call['name'] se match kar raha hai, to hum directly usko invoke kar dete hai with the arguments provided by the LLM
    # result=tool_call['name'].invoke(tool_call['args'])
    # but above line will give error kyuki tool_call['name'] is a string, not the actual function. So we need to map the string to the actual function.
    
    # isliye upar kahi pe hame add_numbers aur subtract_numbers ko define karna pada tha, taki hum unko map kar sakein with the string name of the tool.
    # like this: tools_available={"add_numbers": add_numbers, "subtract_numbers": subtract_numbers}

    result = tools_available[tool_call['name']].invoke(tool_call)
messages.append(result)
messages


Tool called: add_numbers
Arguments: {'a': 5, 'b': 7}
Tool called: subtract_numbers
Arguments: {'a': 12, 'b': 3}


[{'role': 'user', 'content': 'What is the sum of 5 and 7 and subtract 3?'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '222qcyfzj', 'function': {'arguments': '{"a":5,"b":7}', 'name': 'add_numbers'}, 'type': 'function'}, {'id': '5d41x2gtg', 'function': {'arguments': '{"a":12,"b":3}', 'name': 'subtract_numbers'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 290, 'total_tokens': 328, 'completion_time': 0.095029344, 'completion_tokens_details': None, 'prompt_time': 0.101151051, 'prompt_tokens_details': None, 'queue_time': 0.187874307, 'total_time': 0.196180395}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e914b-c8d7-7393-8613-ca3ba31b57ca-0', tool_calls=[{'name': 'add_numbers', 'args': {'a': 5, 'b': 7}, 'id': '222qcyfzj', 'type': 'tool_call'}, {'name': 'subtract